# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [43]:
# Write your code below.

%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [44]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [45]:
import os
from glob import glob
import pandas as pd
# price_data = glob(os.getenv('PRICE_DATA'))
# Write your code below.
# As there are folders inside PRICE_DATA then we should look for parquet files recursivley 
price_data_dir = glob(os.path.join(os.getenv('PRICE_DATA'),'**', "*.parquet"),recursive=True)
price_data_dir



['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2003\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2004\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2004\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2005\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2005\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2006\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2006\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2007\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2007\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2008\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2008\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_200

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [46]:
# Write your code below.
# ddf = dd.read_parquet(price_data_dir).set_index("Ticker")
# ddf = ddf.sort_values(by=["Ticker", "Date"])
dt_list = []
for price_file in price_data_dir:
    # _logs.info(f"Reading file: {s_file}")
    dt = dd.read_parquet(price_file).assign(
        source = os.path.basename(price_file),
        ticker = os.path.basename(price_file).replace('.parquet', ''),
        Date = lambda x: pd.to_datetime(x['Date'])
    )
    dt_list.append(dt)
prices = dd.concat(dt_list, axis = 0, ignore_index = True)
# prices.info()
# prices.head()
# prices = prices.sort_values(by=["ticker", "Date"])
dd_feat = prices.groupby("ticker").apply(
    lambda df: df.assign(
        Close_lag_1=df["Close"].shift(1),
        Adj_Close_lag_1=df["Adj_Close"].shift(1),
        returns=(df["Close"] / df["Close"].shift(1)) - 1,
        hi_lo_range=df["High"] - df["Low"]
    ),
    meta={
        "ticker": "object",
        "Date": "datetime64[ns]",
        "Close": "float64",
        "Adj_Close": "float64",
        "High": "float64",
        "Low": "float64",
        "Close_lag_1": "float64",
        "Adj_Close_lag_1": "float64",
        "returns": "float64",
        "hi_lo_range": "float64"
    }
)
dd_feat.head()

,ticker,Date,Close,Adj_Close,High,Low,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [47]:
print(dd_feat.columns)


Index(['ticker', 'Date', 'Close', 'Adj_Close', 'High', 'Low', 'Close_lag_1',
       'Adj_Close_lag_1', 'returns', 'hi_lo_range'],
      dtype='object')


In [ ]:
# Write your code below.
df_pandas = dd_feat.compute()

df_pandas["returns_ma_10"] = (
    df_pandas.groupby("ticker")["returns"]
    .transform(lambda x: x.rolling(window=10).mean())
)

# Preview the result
df_pandas.head()



Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

If we are working with small to medium-sized data, converting to pandas is fine and often simpler for quick prototyping.
But if our dataset is large or distributed, converting to pandas can be inefficient or even crash your session due to memory limits.

Dask processes chunks of data in parallel, making it ideal for large datasets. It avoids loading everything into RAM. Keeping everything in Dask means we can chain transformations and save to disk without switching contexts.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.